In [46]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col ,to_date, from_unixtime
spark = SparkSession.builder \
    .appName("core") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .getOrCreate()

spark._jsc.hadoopConfiguration().set("fs.s3a.access.key", "admin")
spark._jsc.hadoopConfiguration().set("fs.s3a.secret.key", "admin123")
spark._jsc.hadoopConfiguration().set("fs.s3a.endpoint", "http://minio:9000")
spark._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")
spark._jsc.hadoopConfiguration().set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

In [47]:
df = spark.read.parquet("s3a://raw/assets/")
df.printSchema()
dim_df = df.select(
            col('id').cast('string').alias('assets_id'),
            col('symbol').cast('string'),
            col('name').cast('string').alias('asset_name'),
            col('rank').cast('int'),
            col('priceUsd').cast('decimal(18,8)').alias('price_usd'),
            col('vwap24Hr').cast('decimal(18,8)').alias('change_24h'),
            col('volumeUsd24Hr').cast('decimal(18,8)').alias('market_cap'),
            to_date(from_unixtime(col('date')/1000)).alias('load_dt')
        )
dim_df=dim_df.filter(
                col('priceUsd').isNotNull() & (col('price_usd')>0)
                    )


root
 |-- changePercent24Hr: string (nullable = true)
 |-- explorer: string (nullable = true)
 |-- id: string (nullable = true)
 |-- marketCapUsd: string (nullable = true)
 |-- maxSupply: string (nullable = true)
 |-- name: string (nullable = true)
 |-- priceUsd: string (nullable = true)
 |-- rank: string (nullable = true)
 |-- supply: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- tokens: map (nullable = true)
 |    |-- key: string
 |    |-- value: array (valueContainsNull = true)
 |    |    |-- element: string (containsNull = true)
 |-- volumeUsd24Hr: string (nullable = true)
 |-- vwap24Hr: string (nullable = true)
 |-- date: long (nullable = true)

